ANÁLISIS DATOS CAMPEONATO DE HALTEROFILIA EN ESPAÑA

FASE 1: Análisis con datos proporcionados (2019-2020)

1. Importación y exploración inicial

Webscraping  en la página correspondiente de Wikipedia para obtener los datos de 2019 y 2020

In [ ]:
# Importamos las librerías necesarias para el análisis

import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Obtención de los datos del Campeonato de Halterofilia de 2019

url = "https://es.wikipedia.org/wiki/Campeonato_Europeo_de_Halterofilia_de_2019"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
tablas = pd.read_html(response.text, header=0) #lee todas las tablas de la web
print(f"Se encontraron {len(tablas)} tablas en la página.") #hay 10 tablas en esa página. Hemos de encontrar las que nos interesan (medallistas)
masculino2019 = tablas[5]
femenino2019 = tablas[6]

In [ ]:
masculino2019.head() #ok

In [ ]:
femenino2019.head() #ok

In [ ]:
#conozcamos algo más el contenido de las tablas (missing data, ver si coinciden los campos entre ellas, etc.)
print(masculino2019.info())
print(femenino2019.info())

2. Transformación de datos
Combinación de los datos en un único DataFrame incluyendo Eventos, Año, Género, Medalla (Oro, Plata, Bronce) y Atleta.

In [ ]:
#añadimos género
masculino2019.insert(loc=1, column="Género", value="Masculino")  
femenino2019.insert(loc=1, column="Género", value="Femenino")  

In [ ]:
#compilamos ambas tablas para tener el total de medallistas de 2019 en una sola tabla
total2019=pd.concat([masculino2019, femenino2019], axis=0)
total2019

In [ ]:
#Añadimos año
total2019.insert(loc=1, column="Año", value="2019")  

In [ ]:
#añadimos medalla. Nota: cada columna de Unnamed pertenece al tipo de medalla (oro, plata, bronce) 

total2019.insert(loc=1, column="Medalla Oro", value="Oro")  
total2019.insert(loc=3, column="Medalla Plata", value="Plata")  
total2019.insert(loc=5, column="Medalla Bronce", value="Bronce")  

In [ ]:
#ordenemos la tabla que el tipo de medalla no esté en 3 columnas, si no en 1
total2019_orden = pd.DataFrame({
    "Evento": total2019["Evento"].tolist() * 3,
    "Género": total2019["Género"].tolist() * 3,
    "Año": total2019["Año"].tolist() * 3,
    "Atleta": (
        total2019["Unnamed: 1"].tolist()
        + total2019["Unnamed: 2"].tolist()
        + total2019["Unnamed: 3"].tolist()
      ),
    "Medalla": (
        total2019["Medalla Oro"].tolist()
        + total2019["Medalla Plata"].tolist()
        + total2019["Medalla Bronce"].tolist()
    )
})

In [ ]:
#limpieza de valores no necesarios (p. ej. [n 1])
total2019_orden["Evento"] = total2019_orden["Evento"].str.replace("[n 2]", "", regex=False)
total2019_orden["Atleta"] = total2019_orden["Atleta"].str.replace("[n 1]", "", regex=False)
total2019_orden["Atleta"] = total2019_orden["Atleta"].str.replace("[1]", "", regex=False)
total2019_orden["Atleta"] = total2019_orden["Atleta"].str.replace("[2]", "", regex=False)
total2019_orden #ok, 60 observaciones

In [ ]:
#la variable Atleta tiene varias observaciones, no sólo el nombre del atleta. Quedarnos sólo con nombre y apellidos. 
#Creación del mismo número de elementos en cada fila para poder separar las columnas

total2019_orden["Atleta"] = total2019_orden["Atleta"].str.replace("Reino Unido", "ReinoUnido", regex=False)

total2019_orden["Atleta"] = total2019_orden["Atleta"].str.replace("Van Bellinghen", "Van-Bellinghen", regex=False)

total2019_orden[["Nombre", "Apellidos", "Pais", "Espacio", "Arrancada", "+", "Dos Tiempos", "=", "Total"]]= (total2019_orden["Atleta"].str.split(" ", expand=True))
total2019_orden.head()

In [ ]:
#eliminamos las variables no necesarias
total2019_orden = total2019_orden.drop(columns=["Atleta", "Espacio", "+", "="])

In [ ]:
#dejamos las variables solicitadas
tabla_transformaciondatos= total2019_orden[["Evento", "Año", "Género", "Medalla", "Nombre", "Apellidos"]]
tabla_transformaciondatos = tabla_transformaciondatos.sort_values(by="Apellidos").reset_index(drop=True) #ordenamos por apellido



In [ ]:
#Comprobación final
tabla_transformaciondatos.head() #ok

3. Creación de nuevas columnas

In [ ]:
tabla_nuevascolumnas=total2019_orden[["Nombre", "Apellidos", "Pais", "Arrancada", "Dos Tiempos", "Total"]]
tabla_nuevascolumnas = tabla_nuevascolumnas.sort_values(by="Apellidos").reset_index(drop=True) #ordenamos por apellido

In [ ]:
#Comprobación final
tabla_nuevascolumnas.head() #ok

4. Orden y limpieza

In [ ]:
#Nota, para ello utilizaremos la tabla grande (total2019_orden), que incluye todas las variables limpias y ordenadas

In [ ]:
#creación de variables "Categoría" Y "Fecha" a partir de "Evento"
total2019_orden[["Categoria", "Fecha"]] = total2019_orden["Evento"].str.split(" ", expand=True)
total2019_orden["Fecha"] = total2019_orden["Fecha"].str.replace("(", "")
total2019_orden["Fecha"] = total2019_orden["Fecha"].str.replace(")", "")


In [ ]:
#ordenamos las observaciones por género, categoría y medalla 
tabla_ordenada = total2019_orden.sort_values(by=["Género", "Categoria", "Medalla"]).reset_index(drop=True)
tabla_ordenada = tabla_ordenada.drop(columns=["Evento"]) #es redundante

#para que las categorias '+xxx' no queden de primeras, las renombraremos. Por ejemplo, +87 será 87+
tabla_ordenada["Categoria"] = tabla_ordenada["Categoria"].str.replace("+87", "87+", regex=False)
tabla_ordenada["Categoria"] = tabla_ordenada["Categoria"].str.replace("+109", "109+", regex=False)

#la medalla debería ordenarse por Oro, Plata, Bronce (no alfabéticamente)
orden_medalla = ["Oro", "Plata", "Bronce"] #orden personalizado
tabla_ordenada['Medalla'] = pd.Categorical(tabla_ordenada['Medalla'], categories=orden_medalla, ordered=True)

tabla_ordenada2 = tabla_ordenada.sort_values(by=["Género", "Categoria", "Medalla"]).reset_index(drop=True)

In [ ]:
tabla_ordenada2 #ok

Nota importante: no hay datos del campeonato de Halterofilia de 2020 debido al Covid, por lo que añado ese año al análisis en esta Fase

5. Análisis exploratorio de datos (EDA)

In [ ]:
#exploración de los datos

tabla_ordenada2.describe() #hay 60 observaciones para cada variable, la variable que más valores únicos tiene es Nombre y Apellidos.
#la moda (el valor que más se repite) en este caso es especialmente significativo para las variables numéricas (arrancada, dos tiempos, total...)
#por ejemplo, en la arrancada el valor 101 puntos se repite 3 veces.

In [ ]:
#pasamos variables numéricas a float para poder calcular estadísticas descriptivas
cols = ["Arrancada", "Dos Tiempos", "Total"]
estadisticas = tabla_ordenada2[cols].apply(pd.to_numeric, errors="coerce")
estadisticas.describe().round(2)

In [ ]:
tabla_ordenada2

In [ ]:
tabla_ordenada2.info() #tenemos 11 variables, no hay missing values y casi todas las variables son string (excepto medalla que la hemos vuelto categórica para que se ordene de forma personalizada)

In [ ]:
tabla_ordenada2.Nombre.value_counts() #sólo 6 nombres de 60 se repiten una vez

In [ ]:
tabla_ordenada2.Categoria.value_counts() #la categoria de 55kg está tanto en hombres como en mujeres

In [ ]:
tabla_ordenada2.Fecha.value_counts() #algunos eventos tuvieron lugar el mismo día

In [ ]:
# Cuántas medallas de cada tipo (oro, plata y bronce) ha ganado cada país?
tabla_ordenada2.groupby(["Pais", "Medalla"]).size()

In [ ]:
#gráficamente: medallas totales
conteo = tabla_ordenada2["Pais"].value_counts()
sns.countplot(data=tabla_ordenada2, x="Pais", order=conteo.index)
plt.xticks(rotation=90)  # gira los nombres de países si se superponen
plt.ylabel("Cantidad de Medallas")
plt.title("Medallas por País (2019)")
plt.show()

Rusia es el país que más medallas ha conseguido en total en 2019. Por el contrario, Suecia, Polonia, España, Bélgica, Austria, Francia y Albania son los países que menos medallas han conseguido ese año, sólo una medalla respectivamente

In [ ]:
#gráficamente: por tipo de medalla
conteo = tabla_ordenada2["Pais"].value_counts()
sns.countplot(data=tabla_ordenada2, x="Pais", hue="Medalla", order=conteo.index)
plt.xticks(rotation=90)  # gira los nombres de países si se superponen
plt.ylabel("Cantidad de Medallas")
plt.title("Medallas por País (2019)")
plt.show()

Rusia ha ganado con diferencia más medallas de plata. En oros va empatado con Bielorrusia. Y en bronce con Bielorrusia y Armenia. Suecia, Polonia, España, Bélgica, Austria, Francia y Andorra sólo han conseguido un tipo de medalla

In [ ]:
#¿Qué país ha logrado mayor equidad en el éxito entre sus atletas femeninos y masculinos?
pc2=tabla_ordenada2.groupby(["Pais", "Género"]).size().unstack() #no incluyo el tipo de medalla 
pc2["Equidad"] = pc2.min(axis=1) / pc2.max(axis=1) #así vemos la relación entre el valor máximo de medallas ganadas


In [ ]:
pc2

In [ ]:
pc2["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

In [ ]:
#si lo queremos hacer por tipo de medalla: Oro
pc_oro = tabla_ordenada2[tabla_ordenada2["Medalla"] == "Oro"]
pc2_oro=pc_oro.groupby(["Pais", "Género"]).size().unstack() 
pc2_oro["Equidad"] = pc2_oro.min(axis=1) / pc2_oro.max(axis=1)
pc2_oro["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

In [ ]:
#si lo queremos hacer por tipo de medalla: Plata
pc_plata = tabla_ordenada2[tabla_ordenada2["Medalla"] == "Plata"]
pc2_plata=pc_plata.groupby(["Pais", "Género"]).size().unstack() 
pc2_plata["Equidad"] = pc2_plata.min(axis=1) / pc2_plata.max(axis=1)
pc2_plata["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

In [ ]:
#si lo queremos hacer por tipo de medalla: Bronce
pc_bronce = tabla_ordenada2[tabla_ordenada2["Medalla"] == "Bronce"]
pc2_bronce=pc_bronce.groupby(["Pais", "Género"]).size().unstack() 
pc2_bronce["Equidad"] = pc2_bronce.min(axis=1) / pc2_bronce.max(axis=1)
pc2_bronce["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

Los países donde hay menos equidad son Armenia, Bielorrusia y Rusia, especialmente en medallas de Bronce. Si consideramos la equidad en el total de medallas Italia, Bulgaria y Turquía tampoco tienen equidad de género. El resto, sí.

In [ ]:
tabla_ordenada2.head()

In [ ]:
# ¿Qué país muestra menor diferencia en el promedio de puntos totales obtenidos entre atletas femeninos y masculinos?
tabla_ordenada2["Total"] = pd.to_numeric(tabla_ordenada2["Total"], errors="coerce") #pasar la variable a numérica, porque estaba en string
peso_medio19 = tabla_ordenada2.groupby(["Pais", "Género"])["Total"].mean()
peso_medio19 #ok

In [ ]:
#ahora calculamos la diferencia entre géneros por país
peso_medio19_dif=peso_medio19.unstack()
peso_medio19_dif["Diferencia"] = (peso_medio19_dif["Masculino"] - peso_medio19_dif["Femenino"]). abs() #en valores absolutos
peso_medio19_dif_ordenado = peso_medio19_dif.sort_values("Diferencia") #ordenamos de menor a mayor diferencia
peso_medio19_dif_ordenado #en los casos que aparece NaN es porque no hay medallistas femeninos o masculinos (missing data)

In [ ]:
peso_medio19_dif_ordenado = peso_medio19_dif_ordenado.dropna(subset=["Diferencia"]) #eliminamos los NaN
peso_medio19_dif_ordenado #ok

In [ ]:
#gráficamente
peso_medio19_dif_ordenado["Diferencia"].sort_values(ascending=True).plot(kind="bar", stacked=True)

En 2019 Alemania es el país que menos diferencia mostró en promedio de puntos totales entre atletas de ambos sexos

FASE 2: Web scraping y análisis con datos ampliados (2019-2024)

1. Obtención de datos adicionales


In [ ]:
# Obtención de los datos del Campeonato de Halterofilia de 2021-2024

urls = {
    2021: "https://es.wikipedia.org/wiki/Campeonato_Europeo_de_Halterofilia_de_2021",
    2022: "https://es.wikipedia.org/wiki/Campeonato_Europeo_de_Halterofilia_de_2022",
    2023: "https://es.wikipedia.org/wiki/Campeonato_Europeo_de_Halterofilia_de_2023",
    2024: "https://es.wikipedia.org/wiki/Campeonato_Europeo_de_Halterofilia_de_2024"
}

# Diccionarios para guardar dfs
masculino = {}
femenino= {}
total = {}

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

for year, url in urls.items():
    html = requests.get(url, headers=headers).text 

    tables = pd.read_html(html) #extrae todas las tablas

    # Quedarnos sólo con las tablas que nos interesan(medallistas)
    masculino[year] = tables[2] 
    femenino[year] = tables[3]

     # añadir columna con el año
    masculino[year]["Año"]=year
    femenino[year]["Año"]=year

     # añadir columna con el género
    masculino[year]["Género"]="Femenino"
    femenino[year]["Género"]="Masculino"

     # crear tablas totales (masculino y femenino)
    total[year]={}
    total[year] = pd.concat([masculino[year], femenino[year]], ignore_index=True)

In [ ]:
#comprobación 
total[2021] #ok

In [ ]:
#Unir todas las tablas 2021-2024 en una sola
total21_24 = pd.concat([total[2021], total[2022], total[2023], total[2024]], ignore_index=True)

In [ ]:
#comprobación
total21_24 #ok, 80 observaciones (10 masc + 10 fem * 4 años)

In [ ]:
#añadimos medalla. Nota: cada columna de Unnamed pertenece al tipo de medalla (oro, plata, bronce) 
total21_24.insert(loc=1, column="Medalla Oro", value="Oro")  
total21_24.insert(loc=3, column="Medalla Plata", value="Plata")  
total21_24.insert(loc=5, column="Medalla Bronce", value="Bronce")  

In [ ]:
#ordenemos la tabla que el tipo de medalla no esté en 3 columnas, si no en 1
total21_24_orden = pd.DataFrame({
    "Evento": total21_24["Evento"].tolist() * 3,
    "Género": total21_24["Género"].tolist() * 3,
    "Año": total21_24["Año"].tolist() * 3,
    "Atleta": (
        total21_24["Unnamed: 1"].tolist()
        + total21_24["Unnamed: 2"].tolist()
        + total21_24["Unnamed: 3"].tolist()
      ),
    "Medalla": (
        total21_24["Medalla Oro"].tolist()
        + total21_24["Medalla Plata"].tolist()
        + total21_24["Medalla Bronce"].tolist()
    )
})

In [ ]:
total21_24_orden # ok, 80 filas x 3 tipos de medalla (oro, plata, bronce)

2. Transformación y análisis: integrar los datos obtenidos con los de 2019 y 2020

In [ ]:
total2019_orden.head(3)

In [ ]:
pd.set_option('display.max_rows', None) #para poder ver todas las filas
total21_24_orden

In [ ]:
#limpieza de valores no necesarios (p. ej. [n 1])
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("[n 1]", "", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("[a 1]", "", regex=False)

In [ ]:
#una forma fácil de ver dónde están los nombres o apellidos compuestos
df_split = total21_24_orden["Atleta"].str.split(expand=True) #separar atleta en varias sin limitar por nombre
df_split

In [ ]:
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("+214", "+ 214", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("117+", "117 +", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("103+", "103 +", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("+114", "+ 114", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("171+", " 171 +", regex=False)

total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Reyes Martínez", "Reyes-Martínez", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("García Rincón", "García-Rincón", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Ruiz Velasco", "Ruiz-Velasco", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Hernández Mendoza", "Hernández-Mendoza", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Sánchez López​", "Sánchez-López", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Mata Pérez", "Mata-Pérez", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Brachi García", "Brachi-García", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Lisa Marie", "Lisa-Marie", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Toko Kegne", "Toko-Kegne", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Han Yüksel", "Han-Yüksel", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Sánchez López", "Sánchez-López", regex=False)
total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Sol Anette", "Sol-Anette", regex=False)

total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.replace("Reino Unido", "ReinoUnido", regex=False)


In [ ]:
#veamos cuántas columnas tiene cada fila porque no me dejaba aplicar los nombres, deberían ser 8
total21_24_orden["Atleta"].str.split(" ").apply(len).value_counts() #probablemente espacios en blanco

In [ ]:
longitudes = total21_24_orden["Atleta"].str.split(" ").apply(len)
filas_problema = total21_24_orden[longitudes != 9] #veamos cuál tiene más de 9
print(filas_problema["Atleta"])

total21_24_orden["Atleta"] = total21_24_orden["Atleta"].str.strip().str.replace(r"\s+", " ", regex=True) #quitamos espacios

In [ ]:
#creo las variables adicionales a parte de los nombres y apellidos de los atletas, como en 2019
total21_24_orden[["Nombre", "Apellidos", "Pais", "Arrancada", "+", "Dos Tiempos", "=", "Total"]]= (total21_24_orden["Atleta"].str.split(" ", expand=True))

In [ ]:
total21_24_orden.head()

In [ ]:
#eliminamos las variables no necesarias
total21_24_orden = total21_24_orden.drop(columns=["Atleta", "+", "="])

In [ ]:
#creación de variables "Categoría" Y "Fecha" a partir de "Evento"
total21_24_orden[["Categoria", "Fecha"]] = total21_24_orden["Evento"].str.split(" ", expand=True)
total21_24_orden["Fecha"] = total21_24_orden["Fecha"].str.replace("(", "")
total21_24_orden["Fecha"] = total21_24_orden["Fecha"].str.replace(")", "")


In [ ]:
total2019_orden.head(3)


In [ ]:
#reordenar para que coincida con la de 2019
total21_24_orden = total21_24_orden[["Evento","Género", "Año", "Medalla", "Nombre", "Apellidos", "Pais", "Arrancada", "Dos Tiempos", "Total", "Categoria", "Fecha"]]

In [ ]:
#unimos tablas de 2019 y 2021-2024
total_19_24_orden = pd.concat([total2019_orden, total21_24_orden], ignore_index=True)
total_19_24_orden = total_19_24_orden.drop(columns=["Evento"]) #redundante

In [ ]:
#comprobación
total_19_24_orden.head(3) #ok

In [ ]:
#comprobación2
total_19_24_orden.info() #ok. 300 observaciones (10 masc + 10 fem * 5 años * 3 tipos de medallas)

Preguntas 

In [ ]:
# ¿Cuántas medallas de cada tipo (oro, plata y bronce) ha ganado cada país?
total_19_24_orden.groupby(["Pais", "Medalla"]).size()

In [ ]:
#gráficamente: medallas totales
conteo = total_19_24_orden["Pais"].value_counts()
sns.countplot(data=total_19_24_orden, x="Pais", order=conteo.index)
plt.xticks(rotation=90)  # gira los nombres de países si se superponen
plt.ylabel("Cantidad de Medallas")
plt.title("Medallas por País (2019-2024)")
plt.show()

Cuando miramos el total del periodo, Armenia es el país que más medallas ha conseguido. Ahora Rusia (campeón en 2019) está en 6º lugar. Israel, Serbia, Irlanda y Finlandia son los países que menos medallas han conseguido entre 2019 y 2024.

In [ ]:
#gráficamente: por tipo de medalla
conteo = total_19_24_orden["Pais"].value_counts()
sns.countplot(data=total_19_24_orden, x="Pais", hue="Medalla", order=conteo.index)
plt.xticks(rotation=90)  # gira los nombres de países si se superponen
plt.ylabel("Cantidad de Medallas")
plt.title("Medallas por País (2019-2024)")
plt.show()

Cuando nos fijamos en el tipo de medalla, Armenia es la que más medallas de Oro y Bronce ha ganado, Georgia el que más medallas de plata.

In [ ]:
# ¿Qué país ha logrado mayor equidad en el éxito entre sus atletas femeninos y masculinos?
pc_2=total_19_24_orden.groupby(["Pais", "Género"]).size().unstack() #no incluyo el tipo de medalla 
pc_2["Equidad"] = pc_2.min(axis=1) / pc_2.max(axis=1) #así vemos la relación entre el valor máximo de medallas ganadas
pc_2


In [ ]:
#gráficamente
pc_2["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

En este caso ya hay más inequidad entre países respecto a 2019. Sólo 9 países consiguen una equidad entre hombres y mujeres a la hora de ganar medallas de halterofilia

In [ ]:
#si lo queremos hacer por tipo de medalla: Oro
pc1924_oro = total_19_24_orden[total_19_24_orden["Medalla"] == "Oro"]
pc2_1924_oro=pc1924_oro.groupby(["Pais", "Género"]).size().unstack() 
pc2_1924_oro["Equidad"] = pc2_1924_oro.min(axis=1) / pc2_1924_oro.max(axis=1)
pc2_1924_oro["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

Ucrania es el país con menos equidad en medallas de oro

In [ ]:
#si lo queremos hacer por tipo de medalla: Plata
pc1924_plata = total_19_24_orden[total_19_24_orden["Medalla"] == "Plata"]
pc2_1924_plata=pc1924_plata.groupby(["Pais", "Género"]).size().unstack() 
pc2_1924_plata["Equidad"] = pc2_1924_plata.min(axis=1) / pc2_1924_plata.max(axis=1)
pc2_1924_plata["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

De nuevo, Ucrania es el país con menos equidad en medallas de plata

In [ ]:
#si lo queremos hacer por tipo de medalla: Bronce
pc1924_bronce = total_19_24_orden[total_19_24_orden["Medalla"] == "Bronce"]
pc2_1924_bronce=pc1924_bronce.groupby(["Pais", "Género"]).size().unstack() 
pc2_1924_bronce["Equidad"] = pc2_1924_bronce.min(axis=1) / pc2_1924_bronce.max(axis=1)
pc2_1924_bronce["Equidad"].sort_values(ascending=False).plot(kind="bar", stacked=True)

En este caso, Moldavia es el país con menos equidad en medallas de bronce

In [ ]:
# ¿Qué país muestra menor diferencia en el promedio de puntos totales obtenidos entre atletas femeninos y masculinos?
total_19_24_orden["Total"] = pd.to_numeric(total_19_24_orden["Total"], errors="coerce") #pasar la variable a numérica, porque estaba en string
peso_medio19_24 = total_19_24_orden.groupby(["Pais", "Género"])["Total"].mean()
peso_medio19_24 #ok

In [ ]:
#ahora calculamos la diferencia entre géneros por país
peso_medio19_24_dif=peso_medio19_24.unstack()
peso_medio19_24_dif["Diferencia"] = (peso_medio19_24_dif["Masculino"] - peso_medio19_24_dif["Femenino"]). abs() #en valores absolutos
peso_medio19_24_dif_ordenado = peso_medio19_24_dif.sort_values("Diferencia") #ordenamos de menor a mayor diferencia
peso_medio19_24_dif_ordenado #en los casos que aparece NaN es porque no hay medallistas femeninos o masculinos (missing data)

In [ ]:
peso_medio19_24_dif_ordenado = peso_medio19_24_dif_ordenado.dropna(subset=["Diferencia"]) #eliminamos los NaN
peso_medio19_24_dif_ordenado #ok

In [ ]:
#gráficamente
peso_medio19_24_dif_ordenado["Diferencia"].sort_values(ascending=True).plot(kind="bar", stacked=True)

En el total del periodo 2019-2024 Austria es el país con menor diferencia en promedio de puntos totales entre ambos géneros. 

Como conclusión final, y dado el aumento de países presentados al Campeonato de Halterofilia respecto a 2019 podemos afirmar que hay un interés creciente por esta disciplina. 

** fin del análisis **